# Segmentação de uma pluma de fumaça com o SAM

Modelagem Computacional - Profa Raquel Lobosco
Aluno: Lucas Ribeiro Navarro - RA: 178181

Neste notebook uso o Segment Anything Model (SAM) para segmentar uma pluma de fumaça em uma imagem, vejo onde ele erra e tento melhorar o resultado com caixa e pontos como prompts.

Antes de rodar: Ambiente de execução > Alterar tipo de ambiente de execução > GPU (T4).

In [ ]:
!pip install -q git+https://github.com/facebookresearch/segment-anything.git opencv-python
!wget -q -nc https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

In [ ]:
import numpy as np
import cv2
import torch
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import binary_fill_holes
from google.colab import files
from segment_anything import sam_model_registry, SamPredictor, SamAutomaticMaskGenerator

device = "cuda" if torch.cuda.is_available() else "cpu"
sam = sam_model_registry["vit_b"](checkpoint="sam_vit_b_01ec64.pth")
sam.to(device)
print("rodando em", device)

## 1. Imagem escolhida

Rode a célula abaixo e envie a foto da pluma. Escolhi uma imagem em que a fumaça aparece contra o céu, com nuvens ou prédios por perto, para ter o que avaliar depois.

In [ ]:
enviado = files.upload()
nome_arquivo = list(enviado.keys())[0]
img = cv2.cvtColor(cv2.imread(nome_arquivo), cv2.COLOR_BGR2RGB)

escala = 1024 / max(img.shape[:2])
if escala < 1:
    img = cv2.resize(img, None, fx=escala, fy=escala, interpolation=cv2.INTER_AREA)
H, W = img.shape[:2]
print(W, "x", H)

In [ ]:
def mostrar_grade(imagem, passo=0.1):
    fig, ax = plt.subplots(figsize=(9, 9 * H / W))
    ax.imshow(imagem)
    marcas = np.arange(0, 1.01, passo)
    ax.set_xticks(marcas * W)
    ax.set_xticklabels([f"{m:.1f}" for m in marcas])
    ax.set_yticks(marcas * H)
    ax.set_yticklabels([f"{m:.1f}" for m in marcas])
    ax.grid(color="yellow", alpha=0.5, lw=0.6)
    ax.set_title("Coordenadas relativas (x na horizontal, y na vertical)")
    plt.show()

mostrar_grade(img)

## 2. Máscara gerada pelo SAM (modo automático)

Primeiro rodo o gerador automático, sem nenhum prompt, só para ver o que o SAM entende como objetos na imagem.

In [ ]:
gerador = SamAutomaticMaskGenerator(sam, points_per_side=24, min_mask_region_area=500)
mascaras_auto = gerador.generate(img)
mascaras_auto = sorted(mascaras_auto, key=lambda m: m["area"], reverse=True)
print(len(mascaras_auto), "máscaras encontradas")

rng = np.random.default_rng(0)
cobertura = img.copy().astype(float)
for m in mascaras_auto:
    cor = rng.integers(0, 255, 3)
    cobertura[m["segmentation"]] = 0.45 * cobertura[m["segmentation"]] + 0.55 * cor

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(img)
ax[0].set_title("Original")
ax[1].imshow(cobertura.astype(np.uint8))
ax[1].set_title("Máscaras automáticas")
for a in ax:
    a.axis("off")
plt.show()

## 3. Avaliação do resultado automático

Responder olhando a figura acima:

- A máscara cobre a pluma?
- Inclui regiões que não pertencem a ela (céu, nuvens, prédios)?
- Deixa partes da fumaça de fora?

Escreva aqui: (...)

## 4. Melhorando com prompts

Agora uso o `SamPredictor`, que aceita uma caixa e pontos positivos (dentro da pluma) e negativos (fora dela). As coordenadas são frações da imagem, de 0 a 1, e ajustei olhando a grade da seção 1.

In [ ]:
predictor = SamPredictor(sam)
predictor.set_image(img)

def prever(caixa=None, pos=None, neg=None):
    pos = pos or []
    neg = neg or []
    pontos = rotulos = None
    if pos or neg:
        pontos = np.array([[x * W, y * H] for x, y in pos + neg])
        rotulos = np.array([1] * len(pos) + [0] * len(neg))
    box = None
    if caixa is not None:
        box = np.array([caixa[0] * W, caixa[1] * H, caixa[2] * W, caixa[3] * H])
    mascaras, notas, _ = predictor.predict(
        point_coords=pontos, point_labels=rotulos, box=box, multimask_output=True
    )
    melhor = int(np.argmax(notas))
    return mascaras[melhor], float(notas[melhor])

In [ ]:
caixa_pluma = (0.30, 0.05, 0.80, 0.70)

configs = {
    "A - só caixa": dict(caixa=caixa_pluma),
    "B - caixa + pontos positivos": dict(
        caixa=caixa_pluma,
        pos=[(0.50, 0.40), (0.55, 0.25), (0.45, 0.55)],
    ),
    "C - caixa + positivos + negativos": dict(
        caixa=caixa_pluma,
        pos=[(0.50, 0.40), (0.55, 0.25), (0.45, 0.55)],
        neg=[(0.10, 0.10), (0.90, 0.15), (0.20, 0.80)],
    ),
}

resultados = {}
for nome, cfg in configs.items():
    resultados[nome] = prever(**cfg)
    print(nome, "- nota do SAM:", round(resultados[nome][1], 3))

Os valores de `caixa_pluma`, `pos` e `neg` acima são só um ponto de partida. Depois de ver a grade, troque pelas coordenadas da sua imagem e rode de novo as duas células.

## 5. Comparação lado a lado

Para cada configuração: imagem original com os prompts, máscara binária e máscara sobreposta com contorno.

In [ ]:
def painel(titulo, mascara, cfg):
    fig, ax = plt.subplots(1, 3, figsize=(16, 5.5))

    ax[0].imshow(img)
    c = cfg.get("caixa")
    if c is not None:
        ax[0].add_patch(plt.Rectangle((c[0] * W, c[1] * H), (c[2] - c[0]) * W, (c[3] - c[1]) * H,
                                      fill=False, ec="cyan", lw=2))
    for x, y in cfg.get("pos", []):
        ax[0].plot(x * W, y * H, "o", color="lime", ms=9, mec="k")
    for x, y in cfg.get("neg", []):
        ax[0].plot(x * W, y * H, "o", color="red", ms=9, mec="k")
    ax[0].set_title("Imagem original + prompts")

    ax[1].imshow(mascara, cmap="gray")
    ax[1].set_title("Máscara binária")

    sobre = img.copy()
    sobre[mascara] = (0.5 * sobre[mascara] + 0.5 * np.array([255, 0, 0])).astype(np.uint8)
    contornos, _ = cv2.findContours(mascara.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(sobre, contornos, -1, (255, 255, 0), 2)
    ax[2].imshow(sobre)
    ax[2].set_title("Máscara sobreposta")

    for a in ax:
        a.axis("off")
    fig.suptitle(titulo)
    plt.tight_layout()
    plt.show()

for nome, cfg in configs.items():
    painel(nome, resultados[nome][0], cfg)

Comentário sobre cada configuração (o que melhorou ou piorou quando adicionei a caixa, os pontos positivos e os negativos):

A: (...)

B: (...)

C: (...)

### Limpeza da máscara

Como a apresentação da máscara também conta, aplico abertura e fechamento morfológico, fico só com o maior componente conexo e preencho buracos.

In [ ]:
def limpar(mascara, k=9):
    m = mascara.astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, kernel)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, kernel)
    n, rotulo, estat, _ = cv2.connectedComponentsWithStats(m)
    if n > 1:
        maior = 1 + int(np.argmax(estat[1:, cv2.CC_STAT_AREA]))
        m = (rotulo == maior).astype(np.uint8)
    return binary_fill_holes(m).astype(bool)

def iou(a, b):
    uniao = (a | b).sum()
    return (a & b).sum() / uniao if uniao else 0.0

escolhida = list(configs)[-1]
limpas = {nome: limpar(r[0]) for nome, r in resultados.items()}
ref = limpas[escolhida]

tabela = pd.DataFrame({
    "nota SAM": [resultados[n][1] for n in configs],
    "área bruta (%)": [100 * resultados[n][0].mean() for n in configs],
    "área limpa (%)": [100 * limpas[n].mean() for n in configs],
    "IoU com a escolhida": [iou(limpas[n], ref) for n in configs],
}, index=list(configs)).round(3)
tabela

In [ ]:
painel("Máscara final (" + escolhida + ", com limpeza)", ref, configs[escolhida])

## 6. O que dificultou a segmentação

Aqui meço na própria imagem o que pode ter atrapalhado: brilho e contraste da fumaça em relação à faixa ao redor, diferença de cor e força do gradiente na borda da máscara. Se quiser comparar com uma nuvem, defina `regiao_nuvem` com uma caixa (x0, y0, x1, y1) sobre ela.

In [ ]:
regiao_nuvem = None

cinza = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(float)
hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV).astype(float)
lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB).astype(float)

kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (41, 41))
anel = cv2.dilate(ref.astype(np.uint8), kernel).astype(bool) & ~ref

def resumo(regiao):
    return {
        "brilho médio": cinza[regiao].mean(),
        "contraste interno (desvio)": cinza[regiao].std(),
        "saturação média": hsv[..., 1][regiao].mean(),
    }

dentro = resumo(ref)
fora = resumo(anel)
comparacao = pd.DataFrame({"dentro da máscara": dentro, "faixa ao redor": fora}).round(2)
print(comparacao)

dist_cor = np.linalg.norm(lab[ref].mean(axis=0) - lab[anel].mean(axis=0))
print("\ndistância de cor (Lab) entre pluma e entorno:", round(dist_cor, 2))

gx = cv2.Sobel(cinza, cv2.CV_64F, 1, 0, ksize=3)
gy = cv2.Sobel(cinza, cv2.CV_64F, 0, 1, ksize=3)
grad = np.hypot(gx, gy)
borda = cv2.morphologyEx(ref.astype(np.uint8), cv2.MORPH_GRADIENT, np.ones((3, 3), np.uint8)) > 0
print("gradiente médio na borda:", round(grad[borda].mean(), 2))
print("gradiente médio na imagem:", round(grad.mean(), 2))
print("razão borda/imagem:", round(grad[borda].mean() / grad.mean(), 2))

if regiao_nuvem is not None:
    x0, y0, x1, y1 = regiao_nuvem
    caixa_nuvem = np.zeros((H, W), bool)
    caixa_nuvem[int(y0 * H):int(y1 * H), int(x0 * W):int(x1 * W)] = True
    print("\nfração da região de nuvem dentro da máscara:", round((caixa_nuvem & ref).sum() / caixa_nuvem.sum(), 3))
    d_nuvem = np.linalg.norm(lab[ref].mean(axis=0) - lab[caixa_nuvem].mean(axis=0))
    print("distância de cor (Lab) entre pluma e nuvem:", round(d_nuvem, 2))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(cinza[ref], bins=50, alpha=0.6, density=True, label="pluma")
ax[0].hist(cinza[anel], bins=50, alpha=0.6, density=True, label="entorno")
ax[0].set_xlabel("brilho (0 a 255)")
ax[0].legend()
ax[0].set_title("Histograma de brilho")
ax[1].imshow(grad, cmap="magma")
ax[1].contour(ref, levels=[0.5], colors="cyan", linewidths=1)
ax[1].set_title("Gradiente e contorno da máscara")
ax[1].axis("off")
plt.tight_layout()
plt.show()

Análise (usar os números e gráficos acima):

- Iluminação: (...)
- Contraste entre fumaça e fundo: (...)
- Semelhança com nuvens: (...)
- Bordas da pluma (gradiente baixo = borda difusa): (...)

## Conclusão

O que funcionou melhor, o que ainda erra e o que eu tentaria em seguida: (...)

In [ ]:
cv2.imwrite("mascara_pluma.png", ref.astype(np.uint8) * 255)
sobre = img.copy()
sobre[ref] = (0.5 * sobre[ref] + 0.5 * np.array([255, 0, 0])).astype(np.uint8)
cv2.imwrite("sobreposicao_pluma.png", cv2.cvtColor(sobre, cv2.COLOR_RGB2BGR))
files.download("mascara_pluma.png")
files.download("sobreposicao_pluma.png")